# Exercise 5: Morphophonological rule writing

*Morphological Structure · version 0.1.0*

> Write a formal rule (in prose or as a conditional) for Turkish consonant alternation in the locative suffix (*-da/-ta*): state the trigger (what phonological property of the preceding consonant) and the two output forms. Then apply your rule to the following stems: *el* (hand), *kap* (lid), *göl* (lake), *at* (horse), *iç* (inside).

**Setup.** `pip install "turkicnlp[hfst]==0.5.0"`. Where the compiled `hfst` package has no wheel for your Python version, install `turkicnlp==0.5.0 pyhfst` instead: the first code cell then uses the pure-Python `pyhfst` reader, which has the same lookup interface. The Apertium transducers are downloaded once with `turkicnlp.download`. With `pyhfst`, readings may be listed in a different order than with `hfst`, and the pipeline may select a different reading when several have the same weight.

**Rule.** The locative is *-DA*. Its consonant is *t* if the stem ends in a voiceless consonant (*ç, f, h, k, p, s, ş, t*), and *d* otherwise, that is after a voiced consonant or a vowel. Its vowel follows two-way harmony (*a* after a back last vowel, *e* after a front one). As a conditional: IF final(stem) ∈ {ç, f, h, k, p, s, ş, t} THEN D → t ELSE D → d.

In [ ]:
import sys
try:
    import hfst                                   # compiled HFST, as in the chapter
except ImportError:                               # fall back to the pure-Python reader
    import pyhfst
    sys.modules["hfst"] = pyhfst

import turkicnlp
from turkicnlp import Pipeline

def pipeline(lang):
    turkicnlp.download(lang, processors=["morph"])          # Apertium transducers (once)
    return Pipeline(lang, processors=["tokenize", "morph"], tokenize_backend="rule", morph_backend="apertium")

def analyser(pipe):
    """The Apertium processor of a pipeline (private API in 0.5.0)."""
    return next(p for p in pipe._processors if hasattr(p, "_analyze"))

def readings(pipe, word, plus=False):
    """All transducer readings of one word; by default without '+' continuations (copula etc.)."""
    out = [r["raw"] for r in analyser(pipe)._analyze(word)]
    return out if plus else [r for r in out if "+" not in r]

def show(pipe, sentence):
    for w in pipe(sentence).words:
        print(f"  {w.text:<18} lemma={w.lemma or '_':<10} upos={w.upos or '_':<6} feats={w.feats or '_'}")

tur = pipeline("tur")

In [ ]:
VOICELESS = set("çfhkpsşt")
BACK = set("aıou")

def locative(stem):
    """IF the stem ends in a voiceless consonant THEN -tA ELSE -dA; A = a after a back last vowel, else e."""
    last = next(c for c in reversed(stem) if c in "aıoueiöü")
    return stem + ("t" if stem[-1] in VOICELESS else "d") + ("a" if last in BACK else "e")

for stem in ["el", "kap", "göl", "at", "iç"]:
    form = locative(stem)
    rs = readings(tur, form)
    print(f"{stem:4} -> {form:6} locative reading: {any('<loc>' in r for r in rs)}; all readings: {rs}")

## Worked solution

Output of `experiments/ch05_exercises/run_ch05_exercises.py` on 19 October 2026 (TurkicNLP 0.5.0, HFST 3.16.0.1, Python 3.11):

```
el   -> elde   locative reading: True; all readings: ['el<n><loc>']
kap  -> kapta  locative reading: True; all readings: ['kap<n><loc>', 'kap<v><tv><caus><pot>']
göl  -> gölde  locative reading: True; all readings: ['göl<n><loc>']
at   -> atta   locative reading: True; all readings: ['at<n><loc>', 'at<v><tv><caus><pot>']
iç   -> içte   locative reading: True; all readings: ['iç<n><loc>', 'iç<v><iv><caus><pot>', 'iç<v><tv><caus><pot>']
```

The rule gives *elde*, *kapta*, *gölde*, *atta* and *içte*: *l* is voiced (→ *d*), *p*, *t* and *ç* are voiceless (→ *t*); *a* follows the back vowels of *kap* and *at*, *e* the front vowels of *el*, *göl* and *iç*. The transducer accepts all five as locatives.

For three of them it also returns a verb reading: *kapta*, *atta* and *içte* can be read as verb stem + causative *-t* + potential *-a* (`<caus><pot>`), a stem that occurs only before further suffixes. A rule that generates the locative cannot tell which reading a corpus token has; that needs the disambiguation discussed at the end of the chapter's analysis section.

## Your turn — extension

1. Extend the rule to the ablative *-DAn* and the past tense *-DI* (four-way harmony), and test both on the same stems.
2. Write the rule as a two-level or replace rule in the notation of Finite-State Morphology.